In [1]:
# 📦 Install dependencies
# !pip install plotly numpy pandas matplotlib
# !pip install --upgrade nbformat
# !pip install pySankey seaborn

In [2]:
# 📁 Define project paths
OriginalProjectPath = "../../2018 FreudMeOutProject"
SecondProjectPath = "../../2019 Freud2.0"
MobileProjectPath = "../../2020 Mobile"
VrProjectPath = "../../2023 Affective Game VR"
TestProjectPath = "../."

In [3]:
# 🔤 Extract project name from path
def getProjectNameFromPath(path: str) -> str:
    return path.split("/")[-1]


In [4]:
# 📥 Import libraries
import os
import numpy as np
from collections import defaultdict
import plotly.graph_objects as go


In [5]:
# 🧮 Convert size to human-readable format
def human_readable_size(size_bytes: int) -> str:
    if size_bytes == 0: return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while size_bytes >= 1024 and i < len(units) - 1:
        size_bytes /= 1024.0
        i += 1
    return f"{size_bytes:.1f} {units[i]}"


In [6]:
# 🌲 DirectoryNode class representing folder tree
class DirectoryNode:
    def __init__(self, absolute_path, relative_path="", parent=None):
        self.absolute_path = absolute_path
        self.relative_path = relative_path
        self.parent = parent
        self.children = []
        self.files = []
        self._cached_size = None
        self._cached_file_count = None

    def get_name(self): return os.path.basename(self.absolute_path)
    def add_file(self, file_name): self.files.append(file_name)
    def add_child(self, child_node): self.children.append(child_node)

    def get_size(self):
        if self._cached_size: return self._cached_size
        self._cached_size = sum(os.path.getsize(os.path.join(self.absolute_path, f)) for f in self.files)
        self._cached_size += sum(child.get_size() for child in self.children)
        return self._cached_size

    def get_file_count(self):
        if self._cached_file_count: return self._cached_file_count
        self._cached_file_count = len(self.files) + sum(child.get_file_count() for child in self.children)
        return self._cached_file_count

    def pretty_print(self, indent=0):
        print(" " * indent + f"Node: {self.get_name()} (Size: {human_readable_size(self.get_size())})")
        for file in self.files:
            print(" " * (indent + 2) + f"File: {file}")
        for child in self.children:
            child.pretty_print(indent + 2)


In [7]:
# 🏗️ TreeBuilder: construct directory tree
class TreeBuilder:
    def __init__(self, base_path: str):
        self.base_path = os.path.abspath(base_path)

    def build(self) -> DirectoryNode:
        root = DirectoryNode(self.base_path, os.path.basename(self.base_path))
        nodes = {self.base_path: root}

        for root_dir, dirs, files in os.walk(self.base_path):
            current_node = nodes[root_dir]
            for f in files:
                current_node.add_file(f)
            for d in dirs:
                child_path = os.path.join(root_dir, d)
                rel = os.path.relpath(child_path, self.base_path)
                child_node = DirectoryNode(child_path, rel, parent=current_node)
                current_node.add_child(child_node)
                nodes[child_path] = child_node

        return root


In [8]:
# 🎛️ Display modes for Sankey
class DisplayMode:
    SIZE = "size"
    FILE_COUNT = "file_count"
    CS_FILE_COUNT = "cs_file_count"

In [9]:
# 🔁 Linear value mapping
def map(inmin, inmax, outmin, outmax, value):
    return (value - inmin) / (inmax - inmin) * (outmax - outmin) + outmin


In [10]:
# 📦 SankeyData: structure holding graph data
class SankeyData:
    def __init__(self):
        self.labels = []
        self.source = []
        self.target = []
        self.value = []

    def add_link(self, parent_idx, child_idx, val):
        self.source.append(parent_idx)
        self.target.append(child_idx)
        self.value.append(val)

    def add_label(self, label) -> int:
        self.labels.append(label)
        return len(self.labels) - 1


In [11]:
# ⚙️ Global file label truncate length
TRUNCATE_LABEL_LIMIT = 10

In [12]:
# ✂️ Truncate long file names: VeryLongFileName.cs → VeryLongFileNa(...).cs
def truncate_filename(name: str) -> str:
    if len(name) <= TRUNCATE_LABEL_LIMIT: return name
    base, ext = os.path.splitext(name)
    return base[:TRUNCATE_LABEL_LIMIT] + "(...)" + ext

In [13]:
# 🧠 SankeyGraphBuilder: With .cs file nodes and filename truncation
class SankeyGraphBuilder:
    def __init__(self, display_mode=DisplayMode.FILE_COUNT, max_depth=None, skip_threshold=0.001):
        self.display_mode = display_mode
        self.max_depth = max_depth
        # self.max_depth = max_depth - 1 # Adjusted for root node
        self.skip_threshold = skip_threshold
        self.index_map = {}

    def build(self, root: DirectoryNode) -> SankeyData:
        self.data = SankeyData()
        self._traverse(root)
        return self.data

    def _traverse(self, node: DirectoryNode, depth=0):
        if self._should_skip(node, depth): return
        parent_idx = self._get_or_add_label(node)

        for child in node.children:
            if self._should_skip(child, depth + 1): continue
            child_idx = self._get_or_add_label(child)
            self.data.add_link(parent_idx, child_idx, self._get_value(child))
            self._traverse(child, depth + 1)

        if self.display_mode == DisplayMode.CS_FILE_COUNT:
            if self.max_depth is not None and depth + 1 >= self.max_depth: return
            for file in node.files:
                if not file.endswith(".cs"): continue
                file_label = truncate_filename(file)
                file_idx = self.data.add_label(file_label)
                self.data.add_link(parent_idx, file_idx, 1)


    def _get_or_add_label(self, node: DirectoryNode) -> int:
        rel = node.relative_path or "root"
        if rel in self.index_map: return self.index_map[rel]

        if self.display_mode == DisplayMode.SIZE:
            appendix = human_readable_size(node.get_size())
        elif self.display_mode == DisplayMode.CS_FILE_COUNT:
            appendix = str(self._count_cs_files(node))
        else:
            appendix = str(node.get_file_count())

        label = f"{node.get_name()} ({appendix})"
        idx = self.data.add_label(label)
        self.index_map[rel] = idx
        return idx

    def _get_value(self, node: DirectoryNode) -> int:
        if self.display_mode == DisplayMode.SIZE:
            return node.get_size()
        if self.display_mode == DisplayMode.CS_FILE_COUNT:
            return self._count_cs_files(node)
        return node.get_file_count()

    def _get_parent_value(self, node: DirectoryNode) -> int:
        return self._get_value(node.parent) if node.parent else 1

    def _should_skip(self, node: DirectoryNode, depth: int) -> bool:
        if self.max_depth is not None and depth >= self.max_depth: return True
        if not node.parent: return False
        return self._get_value(node) < self._get_parent_value(node) * self.skip_threshold

    def _count_cs_files(self, node: DirectoryNode) -> int:
        count = len([f for f in node.files if f.endswith(".cs")])
        for child in node.children:
            count += self._count_cs_files(child)
        return count


In [14]:
# 📈 Plot Sankey diagram from SankeyData
def plot_sankey(data: SankeyData, title="Directory Sankey", height=800, width=1800,arrangement="snap", font_size=14):
    fig = go.Figure(go.Sankey(
        arrangement=arrangement,
        node=dict(label=data.labels, align="left"),
        link=dict(source=data.source, target=data.target, value=data.value)
    ))
    fig.update_layout(title_text=title, font_size=font_size, height=height, width=width)
    fig.show()


In [19]:
# 🧪 SecondProjectPath
tree_root = TreeBuilder(SecondProjectPath).build()
graph_size = SankeyGraphBuilder(display_mode=DisplayMode.CS_FILE_COUNT, max_depth=5)
sankey_data_size = graph_size.build(tree_root)
plot_sankey(
    sankey_data_size,
    title=getProjectNameFromPath(SecondProjectPath) + " ( C# files )",
    height=2000,
    width=1500
)

118 files are still hidden

In [16]:
# 🧪 SecondProjectPath
tree_root = TreeBuilder(SecondProjectPath).build()
graph_size = SankeyGraphBuilder(display_mode=DisplayMode.CS_FILE_COUNT, max_depth=4)
sankey_data_size = graph_size.build(tree_root)
plot_sankey(
    sankey_data_size,
    title=getProjectNameFromPath(SecondProjectPath) + " ( C# files )",
    height=1000,
    width=1200,
)

In [20]:
# 🧪 MobileProjectPath
tree_root = TreeBuilder(MobileProjectPath).build()
graph_size = SankeyGraphBuilder(display_mode=DisplayMode.CS_FILE_COUNT, max_depth=7)
sankey_data_size = graph_size.build(tree_root)
plot_sankey(
    sankey_data_size,
    title=getProjectNameFromPath(MobileProjectPath) + " ( C# files )",
    height=800,
    width=1800,
)

In [23]:
# 🧪 VrProjectPath
tree_root = TreeBuilder(VrProjectPath).build()
graph_size = SankeyGraphBuilder(display_mode=DisplayMode.CS_FILE_COUNT, max_depth=6)
sankey_data_size = graph_size.build(tree_root)
plot_sankey(
    sankey_data_size,
    title=getProjectNameFromPath(VrProjectPath) + " ( C# files )",
    height=800,
    width=1800,
)